# 06: 多方法注释与交叉比对

对每个 Leiden 簇并行运行多种标注方法，交叉比对结果。
**多方法并跑不是冗余——不同独立方法的共识最能提高置信度**，
分歧标记需要 PI 重点复核的簇。

本 notebook 产出：
- 各方法独立标注列（`cell_type_{method}_v1`）
- 成对混淆矩阵热图 + Cohen's kappa 表
- 每簇 LLM 综合判决 markdown（需配 API key）
- PI 最终标注列 `cell_type_final_v1`

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：05（多分辨率 Leiden 聚类），读 `05_clustered_v*.h5ad`
- **下游**：06c（亚群 Subset 重分析）/ 07（下游分析），产出 `06_annotated_v*.h5ad`

### 为什么要迭代回跑？
注释质量直接影响下游逐簇分析和亚群重分析的准确性。如果在 06c（亚群分析发现注释不合理）、
07（跨病种比较时发现标签粒度不对）或逐簇报告中发现问题，可能需要：
- 换用不同的 Leiden 分辨率的列（修改 `LEIDEN_COL`）
- 增加或替换标记物 CSV（修改 `MARKER_CSV`）
- 调整 LLM 模型选择（修改 `LLM_MODELS` / `CONSENSUS_MODEL` / `VERDICT_MODEL`）
- 在 PI 手动标注区修改 `marker_assignments` 或 `pi_decisions` 的标签
- 换用 05 的另一个版本（不同 Leiden 分辨率组合）

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `05_clustered_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`
   （例如 `06_annotated_v2.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `LEIDEN_COL`、`MARKER_CSV`、
   `LLM_MODELS` 等），然后 Run All 重跑全部 cell。

> 原始构思（PI）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建、
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

**起点衔接——本 notebook 默认从 05 的最稳定版本出发。**

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH  — 05（已聚类）输出文件路径
# OUTPUT_PATH    — 本 stage 产出 checkpoint 路径
# MARKER_CSV     — 标记物知识库 CSV（供 dotplot + 基因集评分）
# LEIDEN_COL     — 用作簇标签的 obs 列
# RANDOM_SEED    — 随机种子，确保可复现

UPSTREAM_PATH = "results/05_clustered_v1.h5ad"
OUTPUT_PATH   = "results/06_annotated_v1.h5ad"

MARKER_CSV  = "references/markers/gastric_TEST_markers.csv"  # 测试夹具，PI 后续替换为真实 marker 库
LEIDEN_COL  = "leiden_res_0.6"
RANDOM_SEED = 42

# === LLM 配置（读 vault 根 .env 的 LLM_GROUP{N}_* schema）===
# LLM_GROUP  — 使用第几个 LLM group（对应 .env 的 LLM_GROUP{N}_*）。
#              None = 自动用 LLM_DEFAULT_GROUP（当前 .env 里设为 1）。
#              每个 group 包含 1 个 provider + base_url + api_key + 3 档模型。
#              多 group 可用时，mLLMCelltype 共识会自动使用多个端点。
LLM_GROUP = None  # None → 自动取 LLM_DEFAULT_GROUP；手工指定：LLM_GROUP = 1

# LLM 使用的模型级别（每个 group 有 HAIKU / SONNET / OPUS 三档）：
#   "haiku" — 最轻量，适合 mLLMCelltype 初步注释（便宜、快）
#   "sonnet" — 平衡，适合逐簇综合判决（推理-成本平衡）
#   "opus"  — 最强推理，适合争议簇复核（贵、慢，按需启用）
ANNOTATION_MODEL_TIER = "haiku"   # mLLMCelltype 注释用
VERDICT_MODEL_TIER    = "sonnet"  # 逐簇综合判决用

# === scANVI 参考 atlas（不存在则优雅跳过） ===
REFERENCE_ATLAS_PATH = ""  # 留空跳过；填入 .h5ad 路径启用

In [ ]:
# === setup：sys.path + 导入 + 加载上游 adata + 标记物知识库 ===

# 1. 确保框架 src/ 在 sys.path 上，CWD 为项目根目录
import sys, os, gc
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures/06_verdicts", exist_ok=True)
os.makedirs("results/figures/06_sankey", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# 2. 导入（scanpy 原生 API + 框架函数）
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import datetime, warnings

from scrna_integration import load_markers
from scrna_integration.scorers import annotation_concordance

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)

# 3. 加载上游 05 产出
print("加载上游:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"obsm 键: {list(adata.obsm.keys())}")
print(f"leiden 列 '{LEIDEN_COL}': "
      f"{adata.obs[LEIDEN_COL].nunique()} 个簇")

# 4. 加载标记物知识库（marker CSV 不存在时优雅跳过）
if os.path.exists(MARKER_CSV):
    markers = load_markers(MARKER_CSV)
    print(f"标记物库: {MARKER_CSV}")
    print(f"  细胞类型数: {len(markers)}")
    for ct, genes in list(markers.items())[:5]:
        print(f"  {ct}: {genes}")
    if len(markers) > 5:
        print(f"  ... 共 {len(markers)} 种细胞类型")

    # 展平为所有标记基因列表（用于 dotplot）
    all_marker_genes = sorted(set(g for glist in markers.values() for g in glist))
    # 只保留在 adata 中实际存在的基因
    available_markers = [g for g in all_marker_genes if g in adata.var_names]
    missing = set(all_marker_genes) - set(available_markers)
    if missing:
        print(f"  数据中不存在的标记基因（跳过）: {sorted(missing)}")
    print(f"  可用标记基因: {len(available_markers)}/{len(all_marker_genes)}")
else:
    print(f"marker CSV 不存在 ({MARKER_CSV})，跳过标记物知识库加载")
    markers = {}
    available_markers = []

In [ ]:
# === LLM 响应解析 helper（放在这里方便学生看到定义） ===
# 从 LLM 的文本回复中提取 JSON 标注字典，容忍 ```json``` 代码围栏和 thinking 块
import re as _re

def _extract_json_from_llm_response(raw_text: str, expected_clusters: list):
    """从 LLM 原始回复中提取 JSON 标注字典。

    容忍多种格式：纯 JSON、```json ... ``` 代码块、键不带引号等。
    返回 {cluster_id: cell_type} 或空 dict。
    """
    import json as _json

    if not raw_text:
        return {}

    # 策略 1：匹配 ```json ... ``` 代码块
    _match = _re.search(r'```(?:json)?\s*\n?(.*?)\n?```', raw_text, _re.DOTALL)
    if _match:
        _candidate = _match.group(1).strip()
    else:
        # 策略 2：找第一个 { 和最后一个 } 之间的内容
        _start = raw_text.find("{")
        _end = raw_text.rfind("}")
        if _start >= 0 and _end > _start:
            _candidate = raw_text[_start:_end + 1]
        else:
            return {}

    try:
        _result = _json.loads(_candidate)
        if isinstance(_result, dict):
            # 键可能是 int 或 str，统一为 str
            return {str(k): str(v) for k, v in _result.items() if v}
    except (_json.JSONDecodeError, ValueError):
        pass

    return {}

## 方法 1：标记物 dotplot（PI 手动标注）

用已有标记物知识库画 dotplot，每个簇表达哪些标记物一目了然。
**为什么先做这个？** 让 PI 在受自动方法影响前建立自己的判断，避免锚定偏差。
（也可以后做——方法顺序不影响结果，PI 自由选择。）

### 怎么看 dotplot？
- **横轴**：标记基因；**纵轴**：簇
- **颜色深浅**：该基因在该簇的平均表达量
- **圆点大小**：该簇中表达该基因的细胞百分比
- **判断规则**：某个簇对某类细胞的全部标记基因都表达（大圆点 + 深色）
  → 该簇很可能是该细胞类型；只表达个别标记基因 → 可能不是或需更多证据

PI 浏览此图后在下方的 `marker_assignments` 字典中填写每个簇的细胞类型。

In [ ]:
# 标记物 dotplot——每个簇 x 每个标记基因的（表达百分比 + 平均表达量）
if available_markers and LEIDEN_COL in adata.obs.columns:
    sc.pl.dotplot(
        adata, var_names=available_markers, groupby=LEIDEN_COL,
        dendrogram=True, standard_scale="var",
        title=f"Canonical marker dotplot ({LEIDEN_COL})",
        save="_06_dotplot.png",
    )
    # scanpy 默认保存到 figures/，移动到 results/figures/
    src = "figures/dotplot__06_dotplot.png"
    dst = "results/figures/06_dotplot.png"
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"dotplot 已保存: {dst}")
    plt.close("all")
else:
    print("无可用的标记基因或缺少 leiden 列，跳过 dotplot")

In [ ]:
# === PI 手动标注区 ===
# PI 查看上方 dotplot 后，在下方字典中为每个簇 ID 填入细胞类型。
# 示例（基于 Nowicki 数据——PI 需根据实际 dotplot 修改）：
marker_assignments = {
    # "0": "B_cell",
    # "1": "T_cell",
    # "2": "myeloid",
    # ...  PI 逐簇填入
}

if marker_assignments:
    adata.obs["cell_type_marker_v1"] = (
        adata.obs[LEIDEN_COL].astype(str).map(marker_assignments)
    )
    n_assigned = adata.obs["cell_type_marker_v1"].notna().sum()
    print(f"marker 标注: {n_assigned}/{adata.n_obs} 细胞已标注")
else:
    # PI 暂未填写，预建空列保持结构完整
    adata.obs["cell_type_marker_v1"] = np.nan
    adata.obs["cell_type_marker_v1"] = (
        adata.obs["cell_type_marker_v1"].astype("category")
    )
    print("marker 标注: PI 暂未填写，已预建 cell_type_marker_v1 空列")

## 方法 2：LLM 直接注释

每簇取 top 标记基因 → 构造结构化 prompt → LLM 返回 JSON 格式的细胞类型标注。
**为什么用 LLM？** 标记物 dotplot 依靠 PI 领域知识逐簇判断，LLM 提供独立参考——
结合文献知识的推理能识别出仅靠标记物列表难以判定的细胞亚型。

**配置来源**：从 vault 根 `.env` 的 `LLM_GROUP{N}_*` schema 读取，使用 `requests` 直连
（零额外 SDK 依赖，学生可逐行读懂 HTTP 请求）。

**单/多 group 行为**：
- 当前仅 Group1 可用时 → 单模型标注
- PI 后续填入 Group2/3（不同 provider）后 → 逐 group 独立标注 + 多数票共识
- 多 group 间标签不一致的簇自动标记为"争议簇"，供 PI 重点复核

> 技术说明：本 cell 未使用 mLLMCelltype 包。原因是本地 LLM 网关
> 返回的 Anthropic messages 响应中 thinking 块的顺序与 mLLMCelltype 的内部
> 解析器不兼容（后者假定 content[0] 总是 text 块）。改用 `requests`
> 直连后，我们显式提取 text 块，明确容忍 thinking 块的存在。

In [ ]:
# LLM 直接注释——从 vault 根 .env 的 LLM_GROUP schema 读配置
#                          ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 每簇独立调用 LLM（per-cluster，与判决 cell 模式一致）。
# 每个 group 作为独立标注源，多 group 取多数票共识。
# 使用 requests 直连（零额外 SDK 依赖），配置来自 vault 根 .env。
# 无可用 group 时自动跳过。

from scrna_integration.llm_config import load_llm_group_config, get_active_groups
import requests as _requests

# ---- 从 LLM_GROUP schema 读取配置 ----
_active_groups = get_active_groups(project_root=_root)
_has_config = len(_active_groups) > 0

if _has_config:
    # 用默认 group 做注释
    _cfg = load_llm_group_config(group=LLM_GROUP, project_root=_root)
    if _cfg is None:
        _cfg = load_llm_group_config(group=_active_groups[0], project_root=_root)

    _provider = _cfg["provider"]
    _base_url = _cfg["base_url"]
    _api_key = _cfg["api_key"]
    _model = _cfg["models"].get(ANNOTATION_MODEL_TIER, "")
    if not _model:
        _model = (
            _cfg["models"].get("sonnet")
            or _cfg["models"].get("haiku")
            or ""
        )

    print(f"LLM 直接注释配置:")
    print(f"  group: {_cfg['group']} ({_provider})")
    print(f"  base_url: {_base_url}")
    print(f"  model ({ANNOTATION_MODEL_TIER}): {_model}")
    print(f"  活跃 groups: {_active_groups}")
    if len(_active_groups) == 1:
        print("  注意：当前仅 1 个 group 可用，LLM 注释为单模型——"
              "PI 后续填入 Group2/3 后本 cell 会逐 group 独立调用并取共识。")

    # 每簇取 top 标记基因
    sc.tl.rank_genes_groups(
        adata, groupby=LEIDEN_COL, method="wilcoxon",
        n_genes=30, key_added="rank_genes_06",
    )

    # 构建逐簇标记基因字典
    if not hasattr(adata.obs[LEIDEN_COL], "cat") or not pd.api.types.is_categorical_dtype(adata.obs[LEIDEN_COL]):
        adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype("category")
    _cluster_ids = adata.obs[LEIDEN_COL].cat.categories.tolist()
    _marker_dict = {}
    for _cid in _cluster_ids:
        _df = sc.get.rank_genes_groups_df(
            adata, group=_cid, key="rank_genes_06"
        )
        _marker_dict[_cid] = _df["names"].head(10).tolist()
    print(f"为 {len(_marker_dict)} 个簇提取了 top 标记基因")

    # ---- 每簇独立调用 LLM（per-cluster，与判决 cell 模式一致）----
    # 为什么 per-cluster 而非全簇入一个 prompt？
    # ① 全簇 prompt 长，max_tokens=4096 时 thinking 块可能占满预算、截断 text 输出
    # ② 单簇 prompt ~200 tokens，max_tokens=1024 充足（含 thinking），无截断风险
    # ③ 单簇失败不影响其他簇；簇间上下文独立，LLM 更专注
    _all_annotations = []  # [{cluster: cell_type}, ...]，每个 group 贡献一个 dict

    # 确定簇排序（按数字，非数字簇排末尾）
    _sorted_clusters = sorted(
        _marker_dict.keys(),
        key=lambda x: int(x) if str(x).isdigit() else 999
    )

    for _g in _active_groups:
        _gcfg = load_llm_group_config(group=_g, project_root=_root)
        if _gcfg is None:
            continue
        _gprov = _gcfg["provider"]
        _gurl = _gcfg["base_url"]
        _gkey = _gcfg["api_key"]
        _gmodel = _gcfg["models"].get(ANNOTATION_MODEL_TIER, "")
        if not _gmodel:
            _gmodel = (
                _gcfg["models"].get("sonnet")
                or _gcfg["models"].get("haiku")
                or ""
            )
        if not _gmodel:
            continue

        print(f"\n  [{_g}/{len(_active_groups)}] group {_g} ({_gprov}, {_gmodel})")

        # Provider 特化——仅 setup 一次（不随 _cid 循环重复 import/构造）
        if _gprov == "anthropic":
            _api_url = f"{_gurl}/v1/messages"
            _headers_anthropic = {
                "Content-Type": "application/json",
                "x-api-key": _gkey or "placeholder",
                "anthropic-version": "2023-06-01",
            }
        elif _gprov in ("openai", "deepseek", "qwen"):
            from openai import OpenAI
            _oaiclient = OpenAI(api_key=_gkey or "placeholder", base_url=_gurl)
        else:
            print(f"    provider '{_gprov}' 暂未支持，跳过")
            continue

        _group_ann = {}  # {cluster_id: cell_type}

        for _cid in _sorted_clusters:
            _genes_str = ", ".join(_marker_dict[_cid])
            _prompt = (
                f"你是单细胞转录组学专家。以下是胃粘膜 scRNA-seq 数据 "
                f"Leiden 簇 {_cid} 的 top10 标记基因。\n"
                f"请返回该簇最可能的细胞类型名称（仅名称，如 CD4+ Tcm、M1 macrophage）。"
                f"尽量具体到亚型，不确定时标注 best guess。\n\n"
                f"标记基因：{_genes_str}"
            )

            try:
                if _gprov == "anthropic":
                    # Anthropic messages API（requests 直连）
                    _resp = _requests.post(
                        _api_url,
                        headers=_headers_anthropic,
                        json={
                            "model": _gmodel,
                            "messages": [{"role": "user", "content": _prompt}],
                            "system": (
                                "你是单细胞转录组学专家，专精于人胃粘膜细胞图谱注释。"
                                "请只返回细胞类型名称，不要额外解释。"
                            ),
                            "max_tokens": 1024,
                            "temperature": 0.3,
                        },
                        timeout=120,
                    )
                    _data = _resp.json()
                    # 容忍 thinking 块——仅提取 type="text" 的内容
                    _text_blocks = []
                    for _block in _data.get("content", []):
                        if _block.get("type") == "text":
                            _text_blocks.append(_block.get("text", ""))
                    _raw = "\n".join(_text_blocks)
                elif _gprov in ("openai", "deepseek", "qwen"):
                    # OpenAI 兼容 API
                    _resp = _oaiclient.chat.completions.create(
                        model=_gmodel,
                        messages=[
                            {
                                "role": "system",
                                "content": (
                                    "你是单细胞转录组学专家，专精于人胃粘膜细胞图谱注释。"
                                    "请只返回细胞类型名称，不要额外解释。"
                                ),
                            },
                            {"role": "user", "content": _prompt},
                        ],
                        temperature=0.3,
                        max_tokens=1024,
                    )
                    _raw = _resp.choices[0].message.content
                else:
                    print(f"    provider '{_gprov}' 暂未支持，跳过")
                    continue

                # 提取 label：取第一行非空文本
                _label = ""
                for _line in (_raw or "").strip().split("\n"):
                    _clean = _line.strip().strip('"').strip("'").strip()
                    if _clean and not _clean.startswith("`"):
                        _label = _clean
                        break
                if _label:
                    _group_ann[str(_cid)] = _label
                    print(f"    簇 {_cid}: {_label}")
                else:
                    # 降级：整段无换行时取 strip 版本
                    _fallback = (_raw or "").strip().strip('"').strip("'").strip()
                    if _fallback and len(_fallback) < 100:
                        _group_ann[str(_cid)] = _fallback
                        print(f"    簇 {_cid}: {_fallback} (降级)")
                    else:
                        print(f"    簇 {_cid}: 未提取到有效标签（raw_len={len(_raw or '')}）")

            except Exception as _e:
                print(f"    簇 {_cid}: 调用失败: {_e}")
                continue

        if _group_ann:
            _all_annotations.append(_group_ann)
            print(f"  group {_g}: 标注了 {len(_group_ann)} 个簇")
            # 打印标签预览
            _preview = {str(k): v for k, v in sorted(
                _group_ann.items(),
                key=lambda x: int(x[0]) if str(x[0]).isdigit() else 999
            )[:5]}
            print(f"    标签预览: {_preview}")
        else:
            print(f"  group {_g}: 未标注任何簇")

    # ---- 多 group 共识：取多数票（单 group 即自身）----
    if _all_annotations:
        from collections import Counter
        _consensus_map = {}
        _controversial = []
        for _cid in _cluster_ids:
            _votes = [a.get(str(_cid)) for a in _all_annotations]
            _votes = [v for v in _votes if v is not None]
            if not _votes:
                continue
            # 多数票
            _counter = Counter(_votes)
            _top_label, _top_count = _counter.most_common(1)[0]
            _consensus_map[str(_cid)] = _top_label
            # 标记争议簇（各 group 意见不完全一致）
            if len(set(_votes)) > 1:
                _controversial.append(str(_cid))
            # 平票告警：前两名票数相同，most_common(1) 任意选一个
            _top2 = _counter.most_common(2)
            if len(_top2) >= 2 and _top2[0][1] == _top2[1][1]:
                print(
                    f"  平票告警: 簇 {_cid} — {_top2[0][0]} 与 {_top2[1][0]} "
                    f"各 {_top2[0][1]} 票，任意选了 {_top_label}"
                )

        adata.obs["cell_type_llm_v1"] = (
            adata.obs[LEIDEN_COL].astype(str).map(_consensus_map)
        )
        print(f"\nLLM 直接注释完成（per-cluster 模式）:")
        print(f"  groups 参与: {len(_all_annotations)}/{len(_active_groups)}")
        print(f"  共识标注: {len(_consensus_map)} 个簇")
        print(f"  有争议簇 (需 PI 重点复核): {_controversial}")

        adata.uns["cell_type_llm_v1_meta"] = {
            "method": "LLM_GROUP per-cluster annotation",
            "mode": "per-cluster",
            "active_groups": _active_groups,
            "n_groups_responded": len(_all_annotations),
            "model_tier": ANNOTATION_MODEL_TIER,
            "controversial_clusters": _controversial,
            "single_group_note": (
                "仅 1 个 group 可用，LLM 注释为单模型调用。"
                "PI 填入更多 group 后自动启用多模型共识。"
            ) if len(_active_groups) == 1 else None,
            "max_tokens_per_call": 1024,
            "timestamp": datetime.datetime.now().isoformat(),
        }
    else:
        print("\n所有 group 均未返回有效标注。")
        adata.obs["cell_type_llm_v1"] = np.nan
        adata.obs["cell_type_llm_v1"] = (
            adata.obs["cell_type_llm_v1"].astype("category")
        )
else:
    print("=" * 60)
    print("LLM 直接注释已跳过——未检测到可用的 LLM group。")
    print("请在 AI-OS vault 根 .env 文件中配置 LLM_GROUP{N}_*，")
    print("至少填入 LLM_GROUP1_PROVIDER 和 LLM_GROUP1_BASE_URL。")
    print("=" * 60)
    adata.obs["cell_type_llm_v1"] = np.nan
    adata.obs["cell_type_llm_v1"] = (
        adata.obs["cell_type_llm_v1"].astype("category")
    )


## 方法 3：基因集评分

用 `sc.tl.score_genes` 对每类标记基因集合做评分，得到每个细胞相对于每个
细胞类型的连续得分（`obs["score_{celltype}"]`）。

**为什么做基因集评分？** 评分提供了连续性证据——一个簇可能同时高表达多种
细胞类型的标记，说明该簇可能是过渡态或混合群体。评分不直接作为独立标签，而是
作为交叉比对的补充证据：当多个方法对同一簇的标签有分歧时，看该簇对哪种细胞类型的
评分更高，辅助裁决。

### 怎么看评分图？
下方 UMAP 图中，颜色深浅 = 该细胞对该细胞类型的基因集评分（0 到高值）。
某个簇整体颜色深 → 该簇高表达该类标记物 → 支持该细胞类型标注。

In [ ]:
# 对每个细胞类型做基因集评分（sc.tl.score_genes）
# 原理：标记基因平均表达 - 随机参考基因平均表达，产生连续得分
# 用 score_genes 而非 AUCell：scanpy 原生，零额外依赖，学生直接看懂

_score_cols = []
for ct, gene_list in markers.items():
    # 只保留数据中实际存在的基因
    _present = [g for g in gene_list if g in adata.var_names]
    if len(_present) < 2:
        print(f"  {ct}: 可用标记基因 <2（{len(_present)}），跳过评分")
        continue
    col = f"score_{ct}"
    sc.tl.score_genes(
        adata, gene_list=_present, score_name=col,
        ctrl_size=max(1, min(len(_present), 50)),
    )
    _score_cols.append(col)
    print(f"  {ct}: {len(_present)}/{len(gene_list)} 个基因可用 -> obs['{col}']")

print(f"\n共生成 {len(_score_cols)} 个评分列")

In [ ]:
# 逐簇汇总基因集评分——均值 + 阳性细胞百分比
# 单个细胞的评分有噪声，簇级汇总能更稳健地反映该簇的整体标记物信号

if _score_cols and LEIDEN_COL in adata.obs.columns:
    _records = []
    if not hasattr(adata.obs[LEIDEN_COL], "cat") or not pd.api.types.is_categorical_dtype(adata.obs[LEIDEN_COL]):
        adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype("category")
    for _cid in sorted(adata.obs[LEIDEN_COL].cat.categories):
        _mask = adata.obs[LEIDEN_COL] == _cid
        _row = {"cluster": _cid, "n_cells": _mask.sum()}
        for col in _score_cols:
            _vals = adata.obs.loc[_mask, col]
            _ct_name = col.removeprefix("score_")
            _row[f"{_ct_name}_mean"] = round(float(_vals.mean()), 4)
            _row[f"{_ct_name}_pct_pos"] = round(float((_vals > 0).mean()) * 100, 1)
        _records.append(_row)
    _score_summary = pd.DataFrame(_records)
    print("基因集评分逐簇汇总:")
    try:
        from IPython.display import display as ipy_display
        ipy_display(_score_summary)
    except ImportError:
        print(_score_summary.to_string())
    _score_summary.to_csv("results/figures/06_gene_set_scores.csv", index=False)
    print("\n已保存: results/figures/06_gene_set_scores.csv")
else:
    print("无评分列或缺少 leiden 列，跳过逐簇汇总")

In [ ]:
# 基因集评分可视化——UMAP 着色
# 每个子图对应一种细胞类型的评分，颜色深浅 = 该细胞对该类型的得分
# PI 借此判断哪些簇对哪种细胞类型的标记物评分最高

if _score_cols and "X_umap" in adata.obsm:
    n = len(_score_cols)
    n_cols = min(3, n)
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5.5 * n_cols, 4.5 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, col in enumerate(_score_cols):
        ct_name = col.removeprefix("score_")
        ax = axes[i]
        sc.pl.umap(adata, color=col, ax=ax, show=False,
                   title=ct_name, cmap="viridis",
                   vmin=0, vmax="p99", frameon=False)

    # 隐藏多余子图
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    fig.savefig("results/figures/06_gene_set_scores_umap.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("基因集评分 UMAP 已保存: results/figures/06_gene_set_scores_umap.png")
    plt.close("all")
elif not _score_cols:
    print("无评分列，跳过基因集评分可视化")
else:
    print("obsm 中无 X_umap，跳过 UMAP 着色")

## 方法 4：scANVI 标签迁移（守卫——有参考 atlas 时启用）

**前提**：存在对应疾病系统的有标注参考 atlas（含 `cell_type` obs 列）。
scANVI 同时利用参考数据的已知标签和目标数据自身的无监督结构做半监督训练，
标签迁移比简单 kNN 映射更稳健。

当前胃粘膜没有公认的标注参考，**自动跳过**。
后续若 CELLxGENE Census 或合作者提供有标注的胃参考数据，
在 PARAMS 中填入 `REFERENCE_ATLAS_PATH` 即可启用。

In [ ]:
# scANVI 标签迁移（守卫——REFERENCE_ATLAS_PATH 不存在则优雅跳过）
# 为什么用 scANVI 而非直接 kNN 映射？scANVI 同时用参考标签和
# 目标数据无监督结构做半监督训练，标签迁移比简单 kNN 更稳健。

if REFERENCE_ATLAS_PATH and os.path.exists(REFERENCE_ATLAS_PATH):
    import scvi

    print(f"加载参考 atlas: {REFERENCE_ATLAS_PATH}")
    _ref = sc.read_h5ad(REFERENCE_ATLAS_PATH)
    print(f"  参考: {_ref.n_obs:,} 细胞 x {_ref.n_vars:,} 基因")

    # 对齐基因集（参考与目标的交集）
    _common = adata.var_names.intersection(_ref.var_names)
    print(f"  共同基因: {len(_common)}")
    _adata_q = adata[:, _common].copy()
    _ref_sub = _ref[:, _common].copy()

    # 设置 scANVI
    scvi.model.SCANVI.setup_anndata(_adata_q, batch_key="source_dataset")
    scvi.model.SCANVI.setup_anndata(_ref_sub, batch_key="source_dataset")
    _model = scvi.model.SCANVI(
        _ref_sub, _adata_q,
        labels_key="cell_type",
        unlabeled_category="Unknown",
    )
    _model.train(max_epochs=50, early_stopping=True)

    # 预测 + 概率
    _preds = _model.predict(_adata_q)
    adata.obs["cell_type_scanvi_v1"] = _preds["cell_type"].values
    adata.obs["cell_type_scanvi_v1_uncertainty"] = (
        1.0 - _preds["cell_type"].probabilities.max(axis=1)
    )
    adata.uns["scanvi_v1"] = {
        "reference_atlas": REFERENCE_ATLAS_PATH,
        "method": "scANVI",
        "timestamp": datetime.datetime.now().isoformat(),
    }
    print(f"scANVI 完成: {adata.obs['cell_type_scanvi_v1'].nunique()} 类")
else:
    _reason = (
        "未配置 REFERENCE_ATLAS_PATH" if not REFERENCE_ATLAS_PATH
        else f"{REFERENCE_ATLAS_PATH} 不存在"
    )
    print("=" * 60)
    print(f"scANVI 标签迁移已跳过——原因: {_reason}")
    print("如需启用：在 PARAMS 中设置 REFERENCE_ATLAS_PATH 为有标注的 .h5ad 路径，")
    print("并确保参考数据包含 cell_type obs 列。")
    print("=" * 60)
    # 预建空列保持结构完整
    adata.obs["cell_type_scanvi_v1"] = np.nan
    adata.obs["cell_type_scanvi_v1"] = (
        adata.obs["cell_type_scanvi_v1"].astype("category")
    )

## 方法 5（候选，已注释）：CellTypist 预训练分类器

CellTypist 提供多个预训练模型（如 `Immune_All_Low.pkl`、`Developing_Mouse_Brain.pkl` 等）。
**当前注释原因**：PI 的胃/滑膜系统目前没有很好匹配的预训练模型。
当有匹配模型出现时取消下方代码注释即可启用，新列自动进入跨方法比较。

In [ ]:
# === CellTypist（候选——有对应预训练模型时取消注释启用）===
# 前提：pip install celltypist
# 当存在对应组织的 CellTypist 预训练模型时取消注释：
# # import celltypist
# # predictions = celltypist.annotate(
# #     adata, model="Human_Gastric_Atlas.pkl",
# #     majority_voting=True,
# # )
# # adata.obs["cell_type_celltypist_v1"] = (
# #     predictions.predicted_labels["majority_voting"].values
# # )
# # print(f"CellTypist: {adata.obs['cell_type_celltypist_v1'].nunique()} 类")
print(
    "CellTypist 已注释。"
    "当有匹配本组织的预训练模型时取消注释启用。"
)

## 跨方法比较：混淆矩阵热图 + Cohen's kappa + Sankey

对已产生标签的任意两种方法，计算混淆矩阵和 Cohen's kappa，
并用 Sankey 图可视化标签流。

**为什么做跨方法比较？** 这才是多方法并跑的核心价值：
- **一致的方法增强置信度**——两种独立方法给出相同标签 → 可信度高
- **分歧的方法揭示需要 PI 重点复核的簇**——看哪些簇在方法间标签不一致
- **Cohen's kappa 量化一致性**：>0.8 高一致；0.4-0.8 中等一致；<0.4 低一致
- **Sankey 图可视化标签流**——直观展示"方法 A 的 T_cell 被方法 B 拆成了哪些亚型"

### 怎么看这些图？
- **混淆矩阵热图**：行=方法A的标签，列=方法B的标签，颜色深浅=归一化比例。
  对角线亮 = 两方法一致；非对角线亮 = 该标签被方法B重新分类。
- **Cohen's kappa 表**：数值越高越一致，<0.4 的方法对建议 PI 重点复核。
- **Sankey 图**：流向矩阵保存在 `results/figures/06_sankey/`，供外部工具绘制。

In [ ]:
# 收集所有已产生的标注列（不含纯数字、不含空列）
_label_cols = []
for col in sorted(adata.obs.columns):
    if not any(kw in col for kw in ("cell_type", "_v1")):
        continue
    if col.endswith("_proportion") or col.endswith("_entropy"):
        continue
    if col == "cell_type_final_v1":
        continue  # 最终标签由 PI 拍板，不参与双向比较
    # 排除全 NaN 列
    if adata.obs[col].notna().sum() == 0:
        continue
    # 排除数值列（如 uncertainty）
    if adata.obs[col].dtype == "float64" and "uncertainty" in col:
        continue
    _label_cols.append(col)

print(f"可用标注列 ({len(_label_cols)}):")
for col in _label_cols:
    _n = adata.obs[col].nunique()
    print(f"  {col}  ({_n} 类)")
    # 确保是 category 类型
    if adata.obs[col].dtype.name != "category":
        adata.obs[col] = adata.obs[col].astype(str).astype("category")

In [ ]:
# 成对混淆矩阵热图 + Cohen's kappa
# 每个方法对出两张图：(1) 混淆矩阵热图 (2) 打印 kappa 值
from itertools import combinations

if len(_label_cols) >= 2:
    _kappa_results = []
    for _a, _b in combinations(_label_cols, 2):
        # 只比较都有有效标签的细胞
        _valid = adata.obs[_a].notna() & adata.obs[_b].notna()
        if _valid.sum() < 2:
            continue

        # ---- 混淆矩阵热图 ----
        _cm = pd.crosstab(
            adata.obs.loc[_valid, _a].astype(str),
            adata.obs.loc[_valid, _b].astype(str),
            normalize="index",
        )
        fig, ax = plt.subplots(
            figsize=(max(6, len(_cm.columns) * 1.3),
                     max(5, len(_cm.index) * 0.8))
        )
        sns.heatmap(_cm, annot=True, fmt=".2f", cmap="Blues",
                    vmin=0, vmax=1, ax=ax,
                    cbar_kws={"label": "proportion (row-normalized)"})
        ax.set_title(f"{_a}  ->  {_b}")
        ax.set_xlabel(_b)
        ax.set_ylabel(_a)
        plt.tight_layout()
        _safe_a = _a.replace("/", "_").replace(" ", "_")
        _safe_b = _b.replace("/", "_").replace(" ", "_")
        _fname = f"results/figures/06_confusion_{_safe_a}__vs__{_safe_b}.png"
        fig.savefig(_fname, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  混淆矩阵热图: {_fname}")
        plt.close("all")

        # ---- Cohen's kappa ----
        _k = annotation_concordance(adata, label_a=_a, label_b=_b)
        _kappa = _k.get("cohen_kappa", np.nan)
        _kappa_results.append({
            "method_a": _a, "method_b": _b,
            "cohen_kappa": round(float(_kappa), 4),
            "n_cells": int(_valid.sum()),
        })
        print(f"  {_a} vs {_b}: kappa={_kappa:.4f} (n={_valid.sum():,})")

    _kappa_df = pd.DataFrame(_kappa_results)
    if len(_kappa_df) > 0:
        print(f"\nCohen's kappa 汇总:")
        try:
            from IPython.display import display as ipy_display
            ipy_display(_kappa_df)
        except ImportError:
            print(_kappa_df)
        # 成对 kappa 写 CSV
        _kappa_df.to_csv("results/figures/06_kappa_pairs.csv", index=False)
        print("  成对 kappa 表已保存: results/figures/06_kappa_pairs.csv")
        adata.uns["cross_method_comparison_v1"] = {
            "label_columns": _label_cols,
            "kappa_csv": "results/figures/06_kappa_pairs.csv",
            "timestamp": datetime.datetime.now().isoformat(),
        }
else:
    print("可用标注列 <2，跳过成对比较")

In [ ]:
# Sankey 图——可视化两种方法之间的标签流
# 每个节点 = 一种细胞类型标签，连线宽度 = 细胞数
# 为什么用 Sankey？直观展示"方法 A 的 T_cell 被方法 B 拆成了哪些亚型"
# 流向矩阵保存为 CSV，用 R/plotly 等外部工具绘制（matplotlib.sankey 不稳健）

if len(_label_cols) >= 2:
    for _a, _b in combinations(_label_cols, 2):
        _valid = adata.obs[_a].notna() & adata.obs[_b].notna()
        if _valid.sum() < 10:
            continue
        # 构建流向表（method_a -> method_b）
        _flow = pd.crosstab(
            adata.obs.loc[_valid, _a].astype(str),
            adata.obs.loc[_valid, _b].astype(str),
        )
        _safe_a = _a.replace("/", "_").replace(" ", "_")
        _safe_b = _b.replace("/", "_").replace(" ", "_")
        _fname = f"results/figures/06_sankey/flow_{_safe_a}__vs__{_safe_b}.csv"
        _flow.to_csv(_fname)
        print(f"  流向矩阵已保存: {_fname} (shape={_flow.shape})")

    print(f"\nSankey 流向矩阵已保存至 results/figures/06_sankey/")
    print("（用 R/plotly 等工具绘制 Sankey 图）")
else:
    print("可用标注列 <2，跳过 Sankey")

## LLM 逐簇综合判决（读 LLM_GROUP schema——无可用 group 时跳过）

构造包含以下信息的中文 prompt，调用 LLM 逐簇给出综合判决：
- 各方法（marker / LLM 共识 / 基因集评分 / scANVI / CellTypist）对该簇的标签
- Top 标记基因（`sc.tl.rank_genes_groups`）
- 基因集评分 profile
- 各 source_dataset 的 qc_skipped 记录

每簇判决写为独立 markdown 文件：`results/figures/06_verdicts/cluster_{id}.md`

**配置来源**：从 vault 根 `.env` 的 `LLM_GROUP{N}_*` schema 读取。
无可用 group 时优雅跳过，notebook 不崩溃。

In [ ]:
# LLM 逐簇综合判决——从 vault 根 .env 的 LLM_GROUP schema 读配置
#                                ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 每簇独立调用 LLM，避免上下文过长导致输出质量下降。
# 配置来源：项目外（vault 根 .env）→ 安全隔离，不入 git。
# 无可用 group 时优雅跳过，notebook 不崩溃。
import json

from scrna_integration.llm_config import load_llm_group_config, get_active_groups

# ---- 从 LLM_GROUP schema 读取配置 ----
_active_groups = get_active_groups(project_root=_root)
_has_config = len(_active_groups) > 0

if _has_config:
    # 用默认 group（LLM_DEFAULT_GROUP）做逐簇判决
    _cfg = load_llm_group_config(group=LLM_GROUP, project_root=_root)
    if _cfg is None:
        _cfg = load_llm_group_config(group=_active_groups[0], project_root=_root)

    _provider = _cfg["provider"]
    _base_url = _cfg["base_url"]
    _api_key = _cfg["api_key"]
    _model = _cfg["models"].get(VERDICT_MODEL_TIER, "")
    if not _model:
        _model = (
            _cfg["models"].get("sonnet")
            or _cfg["models"].get("haiku")
            or ""
        )

    print(f"LLM 逐簇判决配置:")
    print(f"  group: {_cfg['group']} ({_provider})")
    print(f"  base_url: {_base_url}")
    print(f"  model ({VERDICT_MODEL_TIER}): {_model}")

    # ---- 按 provider 创建 LLM 客户端 ----
    # 支持：anthropic（requests 直连 messages API，零 SDK 依赖）、
    #       openai/deepseek/qwen（openai SDK，复用 OpenAI 兼容协议）
    _client = None  # (provider, url, headers, model) for requests mode
    _use_requests = False  # True → 用 requests 直连；False → 用 openai SDK

    if _provider == "anthropic":
        import requests as _requests
        _client = _requests
        _use_requests = True
        _api_url = f"{_base_url}/v1/messages"
        _headers = {
            "Content-Type": "application/json",
            "x-api-key": _api_key or "placeholder",
            "anthropic-version": "2023-06-01",
        }
        print("  使用 requests 直连 Anthropic messages API（零 SDK 依赖）")

    elif _provider in ("openai", "deepseek", "qwen"):
        from openai import OpenAI
        _client = OpenAI(api_key=_api_key or "placeholder", base_url=_base_url)
        _use_requests = False
        print(f"  使用 OpenAI SDK（{_provider} 兼容 chat/completions）")

    else:
        print(f"  provider '{_provider}' 暂未支持逐簇判决调用。"
              "支持的 provider: anthropic / openai / deepseek / qwen")
        _client = None

    if _client is not None:
        # 确保有 rank_genes_groups 结果（可能方法 2 没跑）
        if "rank_genes_06" not in adata.uns:
            sc.tl.rank_genes_groups(
                adata, groupby=LEIDEN_COL, method="wilcoxon",
                n_genes=30, key_added="rank_genes_06",
            )

        # 收集已有标注列信息
        _annotation_info = []
        for col in adata.obs.columns:
            if not any(kw in col for kw in ("cell_type", "_v1")):
                continue
            if col.endswith("_proportion") or col.endswith("_entropy"):
                continue
            if col == "cell_type_final_v1":
                continue
            _present = adata.obs[col].notna().sum()
            if _present > 0:
                _annotation_info.append(col)

        # 收集基因集评分列
        _score_info = [c for c in adata.obs.columns if c.startswith("score_")]

        # qc_skipped 上下文
        _qc_ctx = adata.uns.get("qc_skipped", {})
        _qc_str = json.dumps(_qc_ctx, ensure_ascii=False, indent=2)

        # 逐簇调用 LLM
        _cluster_ids = sorted(adata.obs[LEIDEN_COL].cat.categories)
        print(f"逐簇 LLM 判决 ({len(_cluster_ids)} 个簇)...")

        for _cid in _cluster_ids:
            # 构建该簇的上下文
            _df = sc.get.rank_genes_groups_df(
                adata, group=_cid, key="rank_genes_06"
            )
            _top_genes = _df["names"].head(15).tolist()
            _top_scores = _df["scores"].head(15).tolist()

            # 各方法对该簇的标签
            _mask = adata.obs[LEIDEN_COL] == _cid
            _method_labels = {}
            for col in _annotation_info:
                _val = adata.obs.loc[_mask, col].mode()
                if len(_val) > 0 and pd.notna(_val.iloc[0]):
                    _method_labels[col] = str(_val.iloc[0])

            # 基因集评分 profile
            _score_profile = {}
            for col in _score_info:
                _score_profile[col] = round(
                    float(adata.obs.loc[_mask, col].mean()), 4
                )

            # 构造中文 prompt
            _gene_lines = "\n".join(
                f"- {g}: logFC={s:.2f}" for g, s in zip(_top_genes, _top_scores)
            )
            _prompt = (
                f"你是单细胞转录组学专家。请判断 Leiden 簇 {_cid} 的细胞类型。\n\n"
                f"## 上下文\n"
                f"- 物种: human\n"
                f"- 组织: stomach（胃粘膜）\n"
                f"- 簇大小: {_mask.sum()} 细胞\n\n"
                f"## 各方法标签\n"
                f"{json.dumps(_method_labels, ensure_ascii=False, indent=2)}\n\n"
                f"## Top 标记基因（按 fold-change 排序）\n"
                f"{_gene_lines}\n\n"
                f"## 基因集评分（簇均值）\n"
                f"{json.dumps(_score_profile, ensure_ascii=False, indent=2)}\n\n"
                f"## 上游数据集 QC 记录\n"
                f"{_qc_str}\n\n"
                f"## 任务\n"
                f"1. 给出最可能的细胞类型（具体到亚型，如 CD4+ Tcm 而非 T_cell）\n"
                f"2. 置信度（high/medium/low）及理由\n"
                f"3. 各方法分歧的原因分析\n"
                f"4. 建议 PI 重点复核什么\n\n"
                f"用中文回答，结构清晰。"
            )

            try:
                if _use_requests:
                    # ---- Anthropic messages API（requests 直连）----
                    _body = {
                        "model": _model,
                        "messages": [{"role": "user", "content": _prompt}],
                        "system": "你是单细胞转录组学专家，专精于人胃粘膜细胞图谱注释。",
                        "max_tokens": 1500,
                        "temperature": 0.3,
                    }
                    _resp = _client.post(
                        _api_url, headers=_headers, json=_body, timeout=120
                    )
                    _data = _resp.json()
                    # 容忍 thinking 块——仅提取 type="text" 的内容
                    _texts = []
                    for _block in _data.get("content", []):
                        if _block.get("type") == "text":
                            _texts.append(_block.get("text", ""))
                        # 跳过 thinking、tool_use 等块
                    _verdict = "\n".join(_texts)
                else:
                    # ---- OpenAI 兼容 API（openai SDK）----
                    _resp = _client.chat.completions.create(
                        model=_model,
                        messages=[
                            {"role": "system",
                             "content": "你是单细胞转录组学专家，专精于人胃粘膜细胞图谱注释。"},
                            {"role": "user", "content": _prompt},
                        ],
                        temperature=0.3,
                        max_tokens=1500,
                    )
                    _verdict = _resp.choices[0].message.content
            except Exception as _e:
                _verdict = f"LLM 调用失败: {_e}"

            # 写 verdict markdown
            _out_path = f"results/figures/06_verdicts/cluster_{_cid}.md"
            with open(_out_path, "w") as _f:
                _f.write(f"# 簇 {_cid} LLM 综合判决\n\n")
                _f.write(f"**模型**: {_model}\n\n")
                _f.write(f"**group**: {_cfg['group']} ({_provider})\n\n")
                _f.write(f"**簇大小**: {_mask.sum()} 细胞\n\n")
                _f.write(f"**各方法标签**:\n")
                for _m, _lbl in _method_labels.items():
                    _f.write(f"- {_m}: {_lbl}\n")
                _f.write(f"\n---\n\n{_verdict}\n")
            print(f"  簇 {_cid}: verdict -> {_out_path}")

        adata.uns["llm_verdict_v1"] = {
            "model": _model,
            "provider": _provider,
            "group": _cfg["group"],
            "base_url": _base_url,
            "output_dir": "results/figures/06_verdicts",
            "n_clusters": len(_cluster_ids),
            "timestamp": datetime.datetime.now().isoformat(),
        }
else:
    print("=" * 60)
    print("LLM 逐簇判决已跳过——未检测到可用的 LLM group。")
    print("请在 AI-OS vault 根 .env 文件中配置 LLM_GROUP{N}_*，")
    print("至少填入 LLM_GROUP1_PROVIDER 和 LLM_GROUP1_BASE_URL。")
    print("=" * 60)

## PI 拍板：`cell_type_final_v1`

PI 阅读每簇的 LLM 判决 markdown 后，手动决定最终标签。
**为什么不让 LLM 自动拍板？** 每个簇的最终标签是科学判断，
LLM 是顾问不是决策者——PI 结合自身领域知识复核后决定。
对胃癌前病变项目（~20-30 簇），阅读所有判决约需 30-60 分钟。

In [ ]:
# === PI 拍板区 ===
# PI 阅读 results/figures/06_verdicts/cluster_{id}.md 后，
# 在下方的 pi_decisions 字典中填入每个簇的最终细胞类型。
# 示例（基于 Nowicki 数据——PI 需修改为实际判断）：
pi_decisions = {
    # "0": "B_cell",
    # "1": "CD4_T_cell",
    # "2": "pit_cell",
    # ...  PI 逐簇填入
}

if pi_decisions:
    adata.obs["cell_type_final_v1"] = (
        adata.obs[LEIDEN_COL].astype(str).map(pi_decisions)
    )
    n_final = adata.obs["cell_type_final_v1"].notna().sum()
    print(f"最终标注: {n_final}/{adata.n_obs} 细胞已标注")
    print(f"  标签种类: {adata.obs['cell_type_final_v1'].nunique()}")
else:
    # PI 暂未填写，预建空列
    adata.obs["cell_type_final_v1"] = np.nan
    adata.obs["cell_type_final_v1"] = (
        adata.obs["cell_type_final_v1"].astype("category")
    )
    print("PI 暂未拍板，已预建 cell_type_final_v1 空列")

In [ ]:
# 记录 PI 拍板的元数据——plain adata.uns 写入，PI 自由记录
adata.uns["cell_type_final_v1_notes"] = {
    "leiden_resolution_used": LEIDEN_COL,
    "available_methods": _label_cols if "_label_cols" in dir() else [],
    "method_basis": "PI manual review of LLM verdicts + marker dotplot",
    "rationale": "PI reviewed each cluster verdict; final labels reflect domain expertise",
    "timestamp": datetime.datetime.now().isoformat(),
}
print("PI 拍板元数据已写入 adata.uns['cell_type_final_v1_notes']")

In [ ]:
# 运行追踪字段——stage + version + upstream
adata.uns["stage"] = "06_annotated"
adata.uns["version"] = "v1"
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"

# 记录 06 运行元数据
adata.uns["06_annotated_v1"] = {
    "upstream": UPSTREAM_PATH,
    "leiden_col": LEIDEN_COL,
    "marker_csv": MARKER_CSV,
    "methods_run": [
        "marker_dotplot",
        "llm_direct_annotation",
        "gene_set_scoring",
    ],
    "scANVI_skipped": not bool(REFERENCE_ATLAS_PATH),
    "timestamp": datetime.datetime.now().isoformat(),
}
print("06 运行元数据已记录")

In [ ]:
# 写入前自检——守卫 adata.X 不变
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("自检通过: X 是 sparse CSR float32")

In [ ]:
# 写出 stage checkpoint（lzf 压缩以节省磁盘空间）
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")

assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 跨 stage 边界释放内存
del adata
gc.collect()
print("内存已释放")